# Predictive Maintenance for Manufacturing Equipment

Predict equipment failure from real-time sensor readings using the **AI4I 2020 Predictive Maintenance Dataset** (10,000 observations, 5 failure modes).

**Business problem** — unplanned downtime is expensive. Flag at-risk machines *before* they fail so operators can perform preventive maintenance. Failure events are rare (3.4 %), so recall on the positive class matters more than headline accuracy.

**Approach**
1. Feature engineering on the 6 raw sensor columns
2. Compare Logistic Regression / Random Forest / Gradient Boosting
3. Tune decision threshold for **F2 score** (weights recall 2× precision)
4. Per-failure-mode breakdown — which modes are learnable from sensors?

In [1]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from predictive_maintenance import (
    load_data, engineer_features, get_feature_matrix,
    train_and_evaluate, per_mode_analysis, FAILURE_MODES, RANDOM_STATE,
)
from sklearn.model_selection import train_test_split

plt.rcParams['figure.dpi'] = 110
pd.options.display.float_format = '{:,.3f}'.format

## 1. Load & explore

In [2]:
df = load_data('ai4i2020.csv')
df.head()

Loaded 10,000 rows, 14 columns
Overall failure rate: 3.39%
  TWF: 46 events (0.46%)
  HDF: 115 events (1.15%)
  PWF: 95 events (0.95%)
  OSF: 98 events (0.98%)
  RNF: 19 events (0.19%)


,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.100,308.600,1551,42.800,0,0,0,0,0,0,0
1,2,L47181,L,298.200,308.700,1408,46.300,3,0,0,0,0,0,0
2,3,L47182,L,298.100,308.500,1498,49.400,5,0,0,0,0,0,0
3,4,L47183,L,298.200,308.600,1433,39.500,7,0,0,0,0,0,0
4,5,L47184,L,298.200,308.700,1408,40.000,9,0,0,0,0,0,0


In [3]:
fig, ax = plt.subplots(figsize=(7, 3))
rates = df[FAILURE_MODES + ['Machine failure']].mean().sort_values()
ax.barh(rates.index, rates.values, color='#c44e52')
for i, v in enumerate(rates.values):
    ax.text(v + 0.0003, i, f'{v:.2%}', va='center', fontsize=9)
ax.set_xlabel('failure rate'); ax.set_title('Class imbalance is severe'); plt.tight_layout(); plt.show()

## 2. Feature engineering

On top of the 6 raw sensor columns, we derive 4 physics-informed features:

| Feature | Formula | Why |
|---|---|---|
| `temp_diff` | process temp − air temp | heat dissipation mechanism |
| `power_proxy` | ω · τ · 2π/60 | motor power ≈ rot × torque |
| `wear_torque` | tool wear × torque | overstrain driver |
| `rpm_per_torque` | rpm / torque | operating regime |

In [4]:
df_fe = engineer_features(df)
X, y, feature_names = get_feature_matrix(df_fe)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE
)
print(f'features: {feature_names}')
print(f'train: {len(y_train):,} ({y_train.mean():.2%} failure)')
print(f'test:  {len(y_test):,} ({y_test.mean():.2%} failure)')

features: ['Type', 'Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]', 'temp_diff', 'power_proxy', 'wear_torque', 'rpm_per_torque']
train: 7,500 (3.39% failure)
test:  2,500 (3.40% failure)


## 3. Train & compare three classifiers

We tune the decision threshold using F2 score (`F2 = (5·P·R) / (4·P + R)`) to reflect the operational cost asymmetry — missing a failure is worse than raising a false alarm.

In [5]:
results, fitted = train_and_evaluate(X_train, X_test, y_train, y_test, feature_names)
results


Logistic Regression
  ROC-AUC: 0.9258
  PR-AUC:  0.4294
  @0.50 -> recall=0.859, precision=0.164
  @0.754 (F2-opt) -> recall=0.729, precision=0.318



Random Forest
  ROC-AUC: 0.9750
  PR-AUC:  0.8509
  @0.50 -> recall=0.729, precision=0.954
  @0.360 (F2-opt) -> recall=0.824, precision=0.854



Gradient Boosting
  ROC-AUC: 0.9807
  PR-AUC:  0.8564
  @0.50 -> recall=0.812, precision=0.932
  @0.211 (F2-opt) -> recall=0.847, precision=0.911


,model,roc_auc,pr_auc,recall_default,precision_default,f1_default,threshold_tuned,recall_tuned,precision_tuned,f1_tuned
0,Logistic Regression,0.926,0.429,0.859,0.164,0.276,0.754,0.729,0.318,0.443
1,Random Forest,0.975,0.851,0.729,0.954,0.827,0.360,0.824,0.854,0.838
2,Gradient Boosting,0.981,0.856,0.812,0.932,0.868,0.211,0.847,0.911,0.878


In [6]:
from sklearn.metrics import precision_recall_curve, average_precision_score, confusion_matrix

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for name, (_, proba, _) in fitted.items():
    prec, rec, _ = precision_recall_curve(y_test, proba)
    ap = average_precision_score(y_test, proba)
    axes[0].plot(rec, prec, label=f'{name} (AP={ap:.3f})')
axes[0].set_xlabel('Recall'); axes[0].set_ylabel('Precision')
axes[0].set_title('Precision-Recall curves'); axes[0].legend(); axes[0].grid(alpha=0.3)

_, _, best_pred = fitted['Gradient Boosting']
cm = confusion_matrix(y_test, best_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Failure', 'Failure'],
            yticklabels=['No Failure', 'Failure'], ax=axes[1])
axes[1].set_title('Confusion Matrix - Gradient Boosting (F2-tuned)'); axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('Actual')
plt.tight_layout(); plt.show()

### Feature importance (Gradient Boosting)

In [7]:
gb = fitted['Gradient Boosting'][0]
importances = pd.Series(gb.feature_importances_, index=feature_names).sort_values()

fig, ax = plt.subplots(figsize=(7, 4))
ax.barh(importances.index, importances.values, color='#4c72b0')
ax.set_title('Feature importance - Gradient Boosting'); ax.set_xlabel('importance')
plt.tight_layout(); plt.show()

## 4. Per-failure-mode breakdown

Fit a separate Random Forest for each of the 5 failure modes. Deterministic physical-mechanism failures should be perfectly learnable from sensor data; random failures should not.

In [8]:
mode_results = per_mode_analysis(pd.read_csv('ai4i2020.csv'), None, feature_names)
mode_results

,failure_mode,n_positive_train,pr_auc,roc_auc
0,TWF,35,0.076,0.968
1,HDF,86,0.982,1.000
2,PWF,71,1.000,1.000
3,OSF,74,0.980,1.000
4,RNF,14,0.009,0.637


**Interpretation**

- **HDF / PWF / OSF** are nearly perfectly separable (PR-AUC 0.98 – 1.00) — they are deterministic functions of the sensor readings.
- **TWF** has some signal (PR-AUC ≈ 0.08) but tool-wear failures are effectively threshold-driven on a single feature that isn't perfectly monotonic in our horizon.
- **RNF** is essentially unlearnable (PR-AUC ≈ 0.01) — by design, random failures carry no sensor signature. This is *useful negative signal*: it tells the plant that no amount of additional sensor ML will help detect this mode; they need different instrumentation or accepting it as residual risk.

## Summary

- **Gradient Boosting** wins: ROC-AUC **0.981**, PR-AUC **0.856**, and at the F2-optimal threshold achieves recall **0.847** at precision **0.911** — i.e. we catch 85 % of true failures with fewer than 1-in-10 false alarms.
- Three of the five failure modes (HDF/PWF/OSF) are essentially fully predictable from the existing sensor suite. A production deployment would issue preventive-maintenance alerts with high confidence for these modes.
- Tool-wear and random failures are *not* reliably predictable from these six sensors — driving sharper instrumentation requirements, not better modelling.